# RAG Fundamentals- Build It From Scratch

**Module 8 · Lesson 8.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Step 1: Chunk Documents
- Steps 2-5: Complete RAG Pipeline in One Class
- RAG With vs Without - See the Difference
- Document Processing - Loaders, Chunking Strategies, Metadata
- Embedding Models - MTEB Rankings, Matryoshka, Multilingual
- Vector Databases - FAISS, Qdrant, Weaviate, Milvus, pgvector

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Chatbot That Invented a Refund Policy

A team shipped a support bot on top of a top-tier LLM. Launch week, a customer asked about refunds:

**📝 `transcript.txt`** - *Intro*

In [ ]:
# Output: the support transcript, reconstructed
#
# User: What is your refund policy?
# Bot:  We offer a 30-day money-back guarantee, no questions asked!
#
# ...except the real policy is 7 days full, 50% up to 30 days, none after.
# The model had never SEEN the company's policy - so it produced a
# confident, fluent, completely INVENTED answer. Support spent the week
# honoring refunds the company never promised.

The model was not broken - it was doing exactly what a closed-book LLM does: answer from its training memory, which does not contain YOUR data. **RAG (Retrieval-Augmented Generation)** fixes this by handing the model the relevant documents at question time: retrieve the real policy, inject it into the prompt, and let the model answer from THAT. Same model, open book, grounded answer - and, crucially, the honesty to say "I do not have that information" when the documents do not cover the question. This lesson builds that pipeline by hand, then maps every production choice that makes it good.

> 📦 **Info**
>
> 🧩 Before you start
> - **Lesson 3.2 (few-shot) & 4.4 (CLIP):** you have already used cosine similarity to compare vectors - RAG's retrieval step is the same idea over document embeddings.
> - **Module 7:** you can call an LLM and read token usage. RAG is one more thing you put in the prompt: retrieved context.
> - **Tools:** Python 3.12+, `pip install google-genai numpy`. A GOOGLE_API_KEY is enough for the whole lesson.

Your linear scan checks every vector for every query. What happens at 4 million chunks?

> 📚 **Analogy**
>
> **RAG is an open-book exam.** Without RAG, the LLM answers from memory - closed-book, might hallucinate. With RAG, it looks up the textbook first. **Same student. Open book. Better answers. No hallucination.**
> **Where this breaks down:** an open book only helps if you can FIND the right page fast - a student flipping randomly still fails. RAG's whole difficulty is the "finding" (retrieval): bad chunks or weak embeddings hand the model the wrong page, and it will confidently answer from it. Most of this lesson is about finding the right page.

The RAG Pipeline - 5 Steps

> 📦 **Info**
>
> RAG vs Fine-Tuning**RAG:** data changes often, no GPU, update = swap docs. **Fine-Tuning:** change model style/skill, needs GPU, update = retrain. **Most companies need RAG, not fine-tuning.**

- Scene 1: closed book - the LLM answers the refund question from memory and INVENTS a policy (the cold open, animated).

- Scene 2: the five-step pipeline - Documents, Chunk, Embed, Retrieve, Augment, Generate - drawn as a conveyor.

- Scene 3: open book - the same question retrieves the real refund chunk, injects it, and the model answers correctly with a citation.

- Scene 4: the unknown question - "blockchain?" retrieves nothing, and grounding makes the model say "I do not have this information" instead of guessing.

Controls: Play/Pause, Reset, speed (0.5x/1x/2x). Scene buttons jump; the caption narrates.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> ⚠️ Misconception check: "Long context windows make RAG obsolete"
> Tempting, since models now take a million tokens. But (1) stuffing your whole corpus into every prompt is slow and expensive - a long-context query can cost dollars where a RAG query costs a cent; (2) models suffer "lost in the middle" - accuracy drops sharply for facts buried in the middle of a huge context; and (3) you lose citations. RAG is not a workaround for small context windows - it is how you feed a model the RIGHT few thousand tokens instead of a irrelevant million. The other trap: "RAG stops hallucination". It only grounds the model IF retrieval surfaces the right chunk; retrieve garbage and the model grounds itself in garbage - "it retrieved something, so the answer must be right" is the anti-pattern to avoid.

## 60-Second Warm-Up: What You Already Know

Two recalls from earlier lessons - both are load-bearing today. Click a card to check yourself.

## Build One: RAG From Scratch in Pure Python

The next three steps build the whole pipeline by hand - chunk, embed, retrieve, augment, generate - in about 60 lines with no framework, so every part is visible. This is the transferable core; the deep dive on each component (steps 4-10) maps the production ecosystem.

---

## 🎯 Concept 1: Step 1: Chunk Documents

### Step 1: Chunk Documents

Split into retrievable pieces with overlap.

Before anything can be retrieved, it has to be cut into retrievable pieces. You cannot embed a 50-page PDF as one vector and expect a precise match, so you split it into chunks - small enough to be specific, with a little overlap so an idea is never severed at a boundary.

A question could be answered by page 30 of a 50-page manual. What has to happen first for RAG to find it?

> ✂️ **Analogy**
>
> **Cutting a long scroll into index cards.** A librarian cannot hand you a whole scroll when you ask one question; they cut it into index cards, each covering one idea, so they can pull the exact card that answers you. The OVERLAP is copying the last sentence of one card onto the top of the next - so a thought that lands on a seam still lives on one whole card.
> **Where this breaks down:** naive word-count cutting slices mid-sentence and mid-table; a card that ends "...the refund is 50" and a next card that starts "percent after 8 days" answers nobody. Step 4 replaces this with boundary-aware (recursive) splitting that cuts at paragraphs and sentences first.

**📝 `01_chunking.py`** - *language-python*

In [ ]:
# RAG STEP 1: CHUNKING
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        if len(chunk.split()) >= 20:
            chunks.append({"id": len(chunks), "text": chunk})
    return chunks

doc = "Netsetos is an edtech company in Hyderabad..." * 20
chunks = chunk_text(doc, chunk_size=60, overlap=10)
print(f"Document: {len(doc.split())} words → {len(chunks)} chunks")
print(f"Sweet spot: 200-500 words, 50-word overlap")

# Output:
#   Document: 120 words -> 3 chunks
#   Sweet spot: 200-500 words, 50-word overlap

- **The step** is `chunk_size - overlap`, so each window starts a bit before the last one ended - that shared strip is the OVERLAP that stops an answer from being split across a boundary.

- **The min-length guard** (`>= 20` words) drops tiny trailing fragments that would just add noise to retrieval.

*Tradeoff:* bigger chunks carry more context but blur the embedding (mixed topics); smaller chunks match precisely but lose surrounding meaning. Step 4 shows recursive and parent-child splitting that beat this naive word-count cut.

#### 💡 What just happened

⚡ What just happened?Documents split into overlapping chunks. Overlap prevents losing info at boundaries. **Chunk size is the #1 tuning parameter: too small = fragments. Too large = noise. Sweet spot: 200-500 words.**

---

## 🎯 Concept 2: Steps 2-5: Complete RAG Pipeline in One Class

### Steps 2-5: Complete RAG Pipeline in One Class

Embed, store, retrieve, augment, generate - all in ~60 lines.

Here is the whole game in one class. Every chunk becomes a vector (embed), the vectors are kept in a list (store), a question is embedded and matched by cosine similarity (retrieve), the top matches are pasted into the prompt (augment), and the model answers from them (generate). Sixty lines, no framework - so nothing is hidden.

"Refund policy" and "how do I get my money back" share zero words. How can retrieval match them?

> 🗡️ **Analogy**
>
> **A clerk who files everything by MEANING.** Imagine a clerk who reads every document and files it not alphabetically but by what it is ABOUT - placing "refund rules" and "how to get my money back" in the same drawer even though they share no words. When you ask a question, the clerk embeds YOUR question the same way, walks to the nearest drawer, and hands the model those pages. The embedding model is that meaning-based filing system; cosine similarity is how the clerk measures "nearest".

**📝 `02_complete_rag.py Full`** - *Pipeline*

In [ ]:
# COMPLETE RAG PIPELINE FROM SCRATCH - no frameworks
# pip install google-genai numpy
from google import genai
from google.genai import types
import numpy as np

client = genai.Client()          # reads GOOGLE_API_KEY
EMBED = "gemini-embedding-001"   # MTEB #1; text-embedding-004 is retired
CHAT  = "gemini-2.5-flash"       # gemini-2.0-flash was shut down 2026-06-01

def embed(text: str, task: str) -> np.ndarray:
    # task_type tunes the vector for its job: documents vs queries.
    r = client.models.embed_content(
        model=EMBED, contents=text,
        config=types.EmbedContentConfig(task_type=task))
    return np.array(r.embeddings[0].values)

class SimpleRAG:
    def __init__(self, chunk_size=60, overlap=10, top_k=3):
        assert overlap < chunk_size, "overlap must be smaller than chunk_size"
        self.chunk_size, self.overlap, self.top_k = chunk_size, overlap, top_k
        self.chunks, self.embeddings = [], []

    def add_document(self, text, source="doc"):
        words = text.split()
        for i in range(0, len(words), self.chunk_size - self.overlap):
            chunk = " ".join(words[i:i + self.chunk_size])
            if len(chunk.split()) < 15:
                continue
            self.chunks.append({"text": chunk, "source": source})
            self.embeddings.append(embed(chunk, "RETRIEVAL_DOCUMENT"))
        print(f"  Indexed: {len(self.chunks)} chunks from '{source}'")

    def retrieve(self, query, k=None):
        k = k or self.top_k
        qe = embed(query, "RETRIEVAL_QUERY")
        # cosine similarity: dot product of unit vectors. Linear scan is fine
        # for a demo; a real corpus uses a vector DB (step 6) for speed.
        scored = [(i, float(qe @ e / (np.linalg.norm(qe) * np.linalg.norm(e))))
                  for i, e in enumerate(self.embeddings)]
        scored.sort(key=lambda x: -x[1])
        return [{"text": self.chunks[i]["text"], "score": s,
                 "source": self.chunks[i]["source"]} for i, s in scored[:k]]

    def query(self, question):
        docs = self.retrieve(question)
        context = "\n\n".join(f"[{d['source']}] {d['text']}" for d in docs)
        prompt = (f"Answer ONLY from this context. If the answer is not in it, say "
                  f"you do not have that information.\n\nContext:\n{context}\n\n"
                  f"Question: {question}\nAnswer:")
        resp = client.models.generate_content(model=CHAT, contents=prompt)
        return {"answer": resp.text.strip(),
                "sources": [{"text": d["text"][:60], "score": f"{d['score']:.3f}"} for d in docs]}

# == BUILD AND TEST ==
rag = SimpleRAG(chunk_size=60, overlap=10, top_k=3)
print("Building RAG...\n")
rag.add_document("Netsetos is an edtech company in Hyderabad offering GenAI, Data "
                 "Science, Cloud courses. Flagship: GenAI Complete Course, 18 "
                 "modules, about 150 hours.", "about.txt")
rag.add_document("Refund policy: Full refund within 7 days. 50 percent from 8-30 "
                 "days. No refunds after 30 days. Processed in 5 business days.", "refund.txt")
rag.add_document("GenAI course: 14999 rupees. Includes lifetime access, Discord, "
                 "weekly live sessions, certificate. EMI via Razorpay.", "pricing.txt")
rag.add_document("Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. "
                 "Python and GCP. 4.8 star rating.", "faq.txt")

print("\nRAG Q&A:\n")
for q in ["What is the refund policy?", "How much does GenAI cost?",
          "Does Netsetos offer blockchain?"]:
    r = rag.query(q)
    print(f"  Q: {q}\n  A: {r['answer'][:120]}\n")

# Output:
#   Indexed: 1 chunks from 'about.txt' ... (4 docs)
#   Q: What is the refund policy?
#   A: Full refund within 7 days, 50 percent from 8-30 days, none after 30 days.
#   Q: Does Netsetos offer blockchain?
#   A: I do not have that information.   <- grounded: it refuses to invent


- **embed(text, task)** - the SAME model embeds documents and queries, but `task_type` ("RETRIEVAL_DOCUMENT" vs "RETRIEVAL_QUERY") tunes each vector for its role; this alone lifts retrieval quality for free.

- **retrieve()** - cosine similarity is just the dot product of unit vectors. The linear scan over a Python list is fine for a demo; a real corpus swaps it for a vector database (step 6) so search stays fast at millions of chunks.

- **query()** - the grounding line ("Answer ONLY from this context...") is the entire anti-hallucination mechanism: it both forces citations and licenses "I do not have that information".

*Why it is only 60 lines:* every RAG framework (LangChain, LlamaIndex - Lessons 8.5, 8.6) is this same five-step loop with more knobs. Build it once by hand and no framework is a black box.

- Scene 1: three phrases each become a point in a small meaning-map.

- Scene 2: "refund policy" and "return my money" land CLOSE despite sharing no words; the unrelated phrase sits far away.

- Scene 3: the query arrow appears; cosine similarity scores nearness by ANGLE, ranking the chunks (0.91, 0.88, 0.20).

- Scene 4: the top-k nearest chunks are pulled into the context tray - that is the whole retrieval step.

Controls: Play/Pause, Reset, speed. This is why keyword search fails and RAG works.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?Complete RAG in ~60 lines: 4 docs chunked, embedded, indexed. Questions matched by cosine similarity, the top few chunks injected into the prompt, Gemini answers from context. The blockchain question returns “I don’t have this information.” **No hallucination. Grounded in YOUR data.**

---

## 🎯 Concept 3: RAG With vs Without - See the Difference

### RAG With vs Without - See the Difference

Same question. With context (accurate) vs without (hallucination).

One question, two prompts. The only difference is whether the real policy is in the context. This is the cold-open incident reduced to five lines of code - and the clearest proof that RAG is not about a smarter model, but about what you put in front of it.

Same model, same question. One prompt includes the real policy, one does not. What differs in the answers?

> 🔑 **Analogy**
>
> **Closed-book vs open-book exam, side by side.** Same student, same question. Closed book, they answer from memory and may bluff a plausible wrong answer. Open book, they read the actual page and answer correctly - and can point at the paragraph. RAG hands the model the page; the "answer ONLY from context" instruction is the exam rule that says "cite the book, do not guess".

**📝 `03_comparison.py`** - *Compare*

In [ ]:
# RAG vs NO RAG - side by side
from google import genai
client = genai.Client()
MODEL = "gemini-2.5-flash"

q = "What is Netsetos's refund policy?"
context = "Refund: Full within 7 days. 50% from 8-30 days. None after 30 days."

no_rag = client.models.generate_content(model=MODEL, contents=q)
with_rag = client.models.generate_content(
    model=MODEL,
    contents=f"Answer ONLY from context. If missing, say so.\nContext: {context}\nQ: {q}")

print(f"WITHOUT RAG:\n  {no_rag.text.strip()[:150]}\n")
print(f"WITH RAG:\n  {with_rag.text.strip()[:150]}\n")

# Output:
#   WITHOUT RAG:
#     I do not have specific information about Netsetos's refund policy...
#     (or worse: a confident, INVENTED policy)
#   WITH RAG:
#     Full refund within 7 days, 50% from 8-30 days, and no refunds after 30 days.
#   Same model. Same question. The only difference is the retrieved context.


> 📦 **Info**
>
> Why each step matters
> - **Chunking:** 50-page PDF too long for prompt. Retrieve ONLY relevant paragraphs.
> - **Embedding:** “Refund policy” and “return my money” have high similarity despite sharing no words.
> - **Retrieval:** Cosine similarity = same technique as CLIP (4.4) and few-shot (3.2).
> - **Augmentation:** “Answer ONLY from context” prevents hallucination.
> - **Generation:** Open-book exam, not a memory test.

| RAG Component | Options | Our Choice | Why |
|---|---|---|---|
| Embedding | OpenAI, Gemini, Cohere | gemini-embedding-001 | MTEB #1, 3072-dim (truncatable), 100+ langs |
| Vector Store | Chroma, Pinecone, pgvector | ChromaDB (dev), pgvector (prod) | Easy start, SQL in production |
| Chunking | Fixed, recursive, semantic | Recursive (500, overlap 50) | Best balance of context and precision |
| Retrieval | Dense, sparse, hybrid | Hybrid (BM25 + dense) | Catches keyword AND semantic matches |
| LLM | Flash, Pro, frontier | Gemini 2.5 Flash | Fast, cheap, good enough for most RAG |

## Go Deeper: Each Component, Production-Grade

Steps 1-3 were the working skeleton. Steps 4-10 open up each stage - document processing, embedding models, vector databases, retrieval strategies, generation, the advanced-RAG family, and India-specific patterns - with the tradeoffs and current tools. Treat it as a map that dates quickly; the ideas (hybrid search, reranking, parent-child chunking) outlast the specific tools.

---

## 🎯 Concept 4: Document Processing - Loaders, Chunking Strategies, Metadata

### Document Processing - Loaders, Chunking Strategies, Metadata

PDF to Markdown. Recursive splitting. Semantic chunking. Parent-child retrieval.

Steps 1-3 chunked clean text. Real corpora are messy PDFs with tables, headings, and columns - and this stage, the least glamorous, moves RAG quality the most. Garbage in (a table shredded across chunks) is garbage retrieved. The winning move: convert to clean Markdown first, then split with structure in mind.

Which stage of RAG most often decides whether the final answer is right?

> 🍳 **Analogy**
>
> **Prepping ingredients before you cook.** A great recipe (your LLM) fails if the vegetables are dumped in whole and unwashed. Document processing is the prep station: peel the PDF into clean Markdown (pymupdf4llm), cut along natural joints not random lengths (recursive splitting), and keep the whole tomato together rather than smearing it across two chunks (keep tables intact). Parent-child is dicing finely to FIND the ingredient, then serving the whole slice for flavor.
> **Where this breaks down:** even perfect prep cannot fix a rotten ingredient - a scanned or legacy-font PDF that extracts as gibberish needs OCR or font conversion FIRST (step 10), before any chunking helps.

| PDF Loader | Speed | Tables | Best For |
|---|---|---|---|
| pymupdf4llm | 0.1s avg | ✅ Markdown | Best default - PDF→Markdown→Chunk |
| pdfplumber | 0.3s | ✅ Best | Table-heavy documents |
| Unstructured.io | 1-5s | ✅ VLM | Complex multi-format (64+ types) |
| Docling (IBM) | 0.5-2s | ✅ TableFormer | Air-gapped deployment, MIT license |

> ✅ **Info**
>
> 📚 Chunking strategies comparison
> - **Recursive Character (recommended default):** tries separators in priority order (paragraph, then newline, then sentence, then word), so it cuts at the most natural boundary that fits. ~512 tokens with ~50 overlap is a strong default; independent 2026 chunking benchmarks put recursive splitting at or near the top of the strategies tested.
> - **Semantic:** splits where the topic shifts (measured by embedding similarity between adjacent sentences). Powerful but risks tiny fragments - always set a minimum chunk size.
> - **Document-structure:** MarkdownHeaderTextSplitter preserves headings. Best for technical docs with clear hierarchy.
> - **Parent-child:** Index small chunks (200-400 chars) for precise matching, return large parent chunks (1000-4000 chars) for context. Solves the precision vs context trade-off.
> - **Contextual (Anthropic):** an LLM prepends a short situating sentence to each chunk before embedding (so "the refund is 50 percent" becomes "In the Netsetos refund policy, the refund is 50 percent..."). Anthropic reports a large drop in retrieval failures; cost is modest with prompt caching. ([Anthropic: Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval))

#### 💡 What just happened

What just happened?Document processing is the most impactful and most overlooked stage. The recommended pipeline: **PDF → pymupdf4llm → Markdown → RecursiveCharacterTextSplitter(512, overlap=50)**. Extract metadata (source, page, section) for filtering. For tables, keep them intact as Markdown - don't chunk across rows. Parent-child retrieval resolves the fundamental tension between retrieval precision (small chunks) and generation context (large chunks).

- Scene 1: why chunk - a 50-page doc must be split so you can retrieve a paragraph, not the whole book.

- Scene 2: naive fixed-size cuts a sentence mid-thought across two chunks, and a query misses the answer.

- Scene 3: overlap keeps the boundary sentence in both neighbors; recursive splitting cuts at paragraph/sentence joints.

- Scene 4: parent-child - match a small chunk for precision, return the large parent for context.

Controls: Play/Pause, Reset, speed. Chunk size is the #1 lever for RAG quality.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Embedding Models - MTEB Rankings, Matryoshka, Multilingual

### Embedding Models - MTEB Rankings, Matryoshka, Multilingual

Gemini Embedding 001 #1. Open-source caught up. Variable dimensions.

The embedding model is the sense of "meaning" your whole system runs on - choose it well, because switching later means re-embedding your entire corpus. The 2026 landscape: Gemini Embedding 001 leads on pure retrieval, open-source (Qwen3, BGE-M3) has caught up, and Matryoshka embeddings let you trade dimensions for cost.

You want cheap fast search AND precise search from ONE embedding. What lets you have both?

> 🎒 **Analogy**
>
> **Matryoshka dolls, for vectors.** A Matryoshka embedding is trained so that any PREFIX of it is still a valid, smaller embedding - like nesting dolls where the small doll inside is a complete doll on its own. So one 3072-dimension vector doubles as a cheap 256-dimension vector: shortlist fast with the small one, rerank precisely with the full one. One embedding, a whole cost/quality dial.
> **Where this breaks down:** only some models are trained this way (Gemini, OpenAI, Cohere, Qwen3) - you cannot just chop a normal embedding in half and expect it to work. And multilingual is a separate axis: for Indian languages, BGE-M3 or Cohere v4 beat higher-ranked English-only models.

| Model | MTEB Score | Dims | Cost/MTok | Key Feature |
|---|---|---|---|---|
| Gemini Embedding 001 | 68.32 | 3072 | $0.15 | Best retrieval score |
| Voyage 3.5 | ~67 | 1024 | $0.06 | Domain models (code, law, finance) |
| Cohere Embed v4 | ~67 | 1536 | $0.12 | Multimodal (text+image) |
| OpenAI 3-small | 62.3 | 1536 | $0.02 | Cheapest commercial option |
| Qwen3-Embedding-8B | 70.58 | 4096 | Free | Best overall (open-source) |
| BGE-M3 | ~65 | 1024 | Free | Dense+sparse+multi-vector, 100+ langs |

> 📦 **Info**
>
> 🎒 Matryoshka embeddings - variable dimensions
> Named after Russian nesting dolls. Embeddings trained so **any prefix is a valid embedding** at that dimensionality. a truncated modern embedding often beats an older full-size one, dimension-for-dimension. even a small prefix of the vector keeps most of the quality, so you can truncate aggressively for cheap first-pass search. Use for two-stage retrieval: shortlist with small embeddings (fast), rerank with full embeddings (precise). Supported by: OpenAI, Gemini, Cohere, Jina, Qwen3.

**📝 `05_matryoshka.py`** - *language-python*

In [ ]:
# Matryoshka: one embedding, a cost/quality dial (google-genai)
from google import genai
from google.genai import types
client = genai.Client()

text = "Netsetos refund policy: full within 7 days."
# Full 3072-dim vector for precise reranking:
full = client.models.embed_content(model="gemini-embedding-001", contents=text).embeddings[0].values
# Truncated 256-dim vector for fast, cheap first-pass search:
small = client.models.embed_content(
    model="gemini-embedding-001", contents=text,
    config=types.EmbedContentConfig(output_dimensionality=256)).embeddings[0].values

print(len(full), len(small))
# Output:
#   3072 256
#   Shortlist with the 256-dim vectors (cheap), then rerank the top hits
#   with the full 3072-dim vectors (precise). Same model, one dial.


#### 💡 What just happened

What just happened?Open-source embeddings (Qwen3, BGE-M3) now **beat commercial options** on benchmarks. Matryoshka embeddings give you a cost/quality dial - use 256 dims for prototypes, 1024 for production. **Critical:** switching embedding models requires re-embedding your entire corpus. Choose carefully and plan migrations. For Indian languages: BGE-M3 or Cohere v4 provide the best cross-lingual retrieval. **The tradeoff:** the highest English-MTEB model is not always the right pick - weigh retrieval score against dimension/cost, multilingual coverage, and the pain of a future re-embed before you commit.

---

## 🎯 Concept 6: Vector Databases - FAISS, Qdrant, Weaviate, Milvus, pgvector

### Vector Databases - FAISS, Qdrant, Weaviate, Milvus, pgvector

Nine options. HNSW vs IVF-PQ. Decision framework by scale.

The Python-list scan in step 2 checks every vector for every query - fine for four chunks, hopeless at four million. A vector database is a specialized index that finds the nearest neighbors in milliseconds without comparing against everything. The choice is mostly about SCALE and ops, not magic.

> 📚 **Analogy**
>
> **A library catalog vs reading every spine.** Your linear scan is walking the whole library reading every book's spine to find one topic. A vector database is the catalog + the way the shelves are arranged (the index) so you jump near the answer instantly. HNSW is a well-signposted browsing path through the stacks (fast, memory-hungry); IVF-PQ compresses the books to fit a bigger library in the same room (billions of vectors); flat is just reading every spine (fine for a tiny shelf).
> **Where this breaks down:** approximate indexes trade a little RECALL for huge speed - they occasionally miss the true nearest book. That is usually fine, but it is why "cosine similarity is correct most of the time" is a real caveat, not a guarantee.

| Database | Type | Best For | Scale | Price |
|---|---|---|---|---|
| FAISS | Library | Custom pipelines, research | Billions (GPU) | Free |
| Chroma | Embedded | Prototypes and small indexes | <10M | Free |
| Qdrant | Server | Production RAG at moderate scale | 100M+ | Free tier 1GB |
| Weaviate | Server | Hybrid search + multi-tenancy | 100M+ | Free sandbox |
| Milvus/Zilliz | Distributed | Billion-scale | Billions | Free tier 5GB |
| Pinecone | Managed | Zero-ops, fastest time-to-prod | 100M+ | Usage-based |
| pgvector | PG extension | Already using PostgreSQL | <5M | Free |
| LanceDB | Embedded | Serverless, S3-backed, edge | 100M+ | Free OSS |

> ✅ **Info**
>
> ⚙️ Indexing algorithms
> - **HNSW (default):** a multi-layer graph giving high recall in about a millisecond, memory-intensive; the usual choice up to roughly a hundred million vectors.
> - **IVF-PQ:** cluster then quantize for a large memory reduction; the choice for billions of vectors where a graph index will not fit in RAM.
> - **ScaNN (Google):** markedly smaller and faster than a graph index, useful when indexes do not fit in RAM.
> - **Flat:** brute-force with perfect recall; fine for a small index (tens of thousands of vectors).

**📝 `06_chroma.py`** - *language-python*

In [ ]:
# The vector-DB upgrade: swap the Python-list scan for Chroma
# pip install chromadb
import chromadb
client = chromadb.Client()
col = client.create_collection("netsetos")

# add() stores documents + their embeddings; Chroma builds the index for you.
col.add(ids=["refund", "pricing"],
        documents=["Full refund within 7 days, 50 percent to 30 days.",
                   "GenAI course is 14999 rupees, EMI via Razorpay."],
        embeddings=[embed(d, "RETRIEVAL_DOCUMENT").tolist()
                    for d in ["Full refund...", "GenAI course..."]])

hits = col.query(query_embeddings=[embed("how do I get my money back", "RETRIEVAL_QUERY").tolist()],
                 n_results=1)
print(hits["ids"][0])
# Output:
#   ['refund']
#   Same retrieve step as SimpleRAG, but Chroma's index stays fast at
#   millions of vectors - no linear scan. This is step 2's list, productionized.


#### 💡 What just happened

What just happened?**Decision framework:** Prototype → Chroma or FAISS. Production <50M vectors → Qdrant (best metadata filtering). Hybrid search priority → Weaviate. Billion-scale → Milvus. Zero ops → Pinecone. Already on PostgreSQL → pgvector. Edge/serverless → LanceDB. **Distance metrics:** cosine similarity is the right default for most text embeddings, but always match the metric the embedding model was trained with (some use dot product or Euclidean).

---

## 🎯 Concept 7: Retrieval Strategies - Hybrid Search, Reranking, Query Transforms

### Retrieval Strategies - Hybrid Search, Reranking, Query Transforms

Dense + sparse hybrid. Cross-encoder reranking. Query transforms.

This is where RAG quality is won or lost, and the upgrades stack in a known order. Dense embeddings alone miss exact terms (a product code, a name); add keyword search, then a precise reranker, then contextual chunks - each independently measurable. One rule governs all of it: the retriever sets the ceiling.

Dense retrieval keeps missing an exact code "GCP-5000". Why, and what is the cheap fix?

> 🏛️ **Analogy**
>
> **A hiring funnel.** Dense retrieval is a resume keyword-scan that surfaces roughly-right candidates but misses the perfect one who phrased things differently - so you add a second channel (BM25 keyword match) and merge the two shortlists fairly (Reciprocal Rank Fusion). Then reranking is the INTERVIEW: too slow to run on everyone, so you interview the top 100 and promote the true best 5. But if the ideal candidate never made the shortlist, no interview can hire them - the sourcing (retrieval) sets the ceiling.
> **Where this breaks down:** more stages = more latency and cost. Measure each addition (with RAGAS, Lesson 8.11) before you pay for it; hybrid search is nearly free and almost always worth it, HyDE and multi-query are not always.

> 📦 **Info**
>
> 🔍 The production retrieval stack
> - **1. Hybrid search (non-negotiable):** dense (embeddings) captures MEANING; sparse (BM25, a classic keyword-frequency score) captures EXACT terms like product codes and names. Merge the two ranked lists with Reciprocal Rank Fusion (`score(d) = Σ 1/(60 + rank_i(d))` - reward items ranked high by either method). Nearly free, and consistently the single biggest quality jump over dense-only.
> - **2. Reranking:** a bi-encoder embeds query and document SEPARATELY (fast, so it can shortlist top-100); a cross-encoder reads the query and document TOGETHER (slow but far more accurate) to re-score that shortlist to a top-5. Cohere Rerank or the free self-hosted BGE-reranker both give a large precision jump on the shortlist.
> - **3. Query transforms:** HyDE (Hypothetical Document Embeddings - the LLM writes a fake ideal answer, and you embed THAT instead of the short question, because an answer looks more like the target documents than a question does). Multi-query (generate several phrasings, merge with RRF). Step-back prompting (abstract the question first). Useful, but with diminishing returns - measure before adding.
> - **4. Contextual retrieval (Anthropic):** Prepend LLM-generated context to chunks. -67% retrieval failure rate. $1.02/M tokens with caching.

#### 💡 What just happened

What just happened?The upgrade stack, in order of bang-for-buck: naive dense, then hybrid search, then reranking, then contextual retrieval - each independently measurable. **The key insight: "the retriever sets the ceiling."** If the correct document is not in the initial candidate set, no reranker can rescue it - so invest in better RETRIEVAL before you invest in a better reranker.

- Scene 1: dense-only retrieval MISSES an exact keyword match (a product code); the quality bar sits low.

- Scene 2: hybrid search adds BM25 keywords and merges with RRF; both the keyword and the semantic match surface. Biggest jump.

- Scene 3: reranking - a cross-encoder re-scores the top-100 shortlist to a precise top-5; the bar jumps again.

- Scene 4: the ceiling - if the right doc was never in the candidates, no reranker can find it. Fix retrieval first.

Controls: Play/Pause, Reset, speed. Replay scene 4 - the retriever sets the ceiling.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: Generation - RAG Prompts, Citations, Context Window Management

### Generation - RAG Prompts, Citations, Context Window Management

Grounding instructions. Source attribution. Stuffing vs map-reduce. Conflicting docs.

Retrieval found the pages; now the model has to answer FROM them, cite them, and handle the awkward cases - conflicting documents, no answer at all, too many chunks to fit. A production RAG prompt is a small contract, not a one-liner.

Two retrieved documents disagree about the refund window. What should a production RAG prompt do?

> ⚖️ **Analogy**
>
> **Briefing an expert witness.** You do not tell a witness "say something true about refunds"; you hand them numbered exhibits and say: answer only from these, cite the exhibit number for every claim, flag it if two exhibits disagree, and if the exhibits do not cover it, say so - do not speculate. That briefing IS the RAG prompt: role, grounding instruction, numbered source blocks, and explicit "I do not know" behavior.
> **Where this breaks down:** witnesses read every exhibit; LLMs get "lost in the middle" and under-weight facts buried in the center of a long context - so put the most relevant chunks at the START and END, and do not stuff more than you need.

> ✅ **Info**
>
> 📝 Production RAG prompt template
> Every RAG prompt needs 4 layers: (1) Role definition, (2) Grounding instruction ("Answer ONLY using provided context"), (3) Numbered context blocks with source IDs, (4) User query + few-shot examples showing citation format AND "I don't know" behavior. Format chunks with XML-style delimiters. Most important: explicitly handle no-answer scenarios - never let the model guess.

| Strategy | How It Works | Best For | Faithfulness |
|---|---|---|---|
| Stuffing | All chunks in one prompt | Simple, a handful of chunks | 1.00 |
| Map-Reduce | Process each separately, combine | Many chunks, parallelizable | 0.95 |
| Refine | Build answer iteratively per chunk | Comprehensive answers | 0.85 |

**📝 `08_rag_prompt.py`** - *language-python*

In [ ]:
# A production RAG prompt: the 4 layers that stop hallucination
def build_prompt(question, chunks):
    numbered = "\n".join(f"[{i+1}] ({c['source']}) {c['text']}"
                         for i, c in enumerate(chunks))
    return f"""You are a support assistant. Answer using ONLY the sources below.
Cite the source number [n] for every claim. If two sources disagree, say so.
If the answer is not in the sources, reply exactly: "I do not have that information."

Sources:
{numbered}

Question: {question}
Answer:"""

# Output: (the model now cites and refuses honestly)
#   Q: What is the refund policy?
#   A: Full refund within 7 days; 50 percent from 8-30 days; none after [1].
#   Q: Do you ship internationally?
#   A: I do not have that information.


#### 💡 What just happened

What just happened?**Citation is non-negotiable** for production RAG. Number source chunks [1], [2], etc. and instruct the model to cite every claim. For conflicting documents: instruct the LLM to note disagreements explicitly rather than silently merging. **When to use which combine strategy:** stuffing is simplest and most faithful for a few chunks; map-reduce trades a little faithfulness for scale when you have many; refine when you need one comprehensive synthesized answer - pick by chunk count and latency budget. Prefer newer documents (temporal filtering) and higher-authority sources. **"Lost in the middle":** LLMs attend most to the START and END of a long context and under-weight the MIDDLE, so accuracy drops sharply for facts buried in the center. Put your most relevant chunks first and last.

---

## 🎯 Concept 9: Advanced RAG - Self-RAG, CRAG, GraphRAG, Agentic RAG

### Advanced RAG - Self-RAG, CRAG, GraphRAG, Agentic RAG

Beyond naive retrieve-and-read. Self-correcting. Knowledge graphs. Agent loops.

"Retrieve once, then read" is naive RAG - and it fails on hard questions (multi-hop, "what are the themes across ALL these docs?", or when the top chunk is wrong). A family of architectures fixes specific failures: self-checking retrieval, corrective fallback, graph-structured knowledge, and agent loops that retrieve iteratively.

A question needs facts from THREE different documents combined. Where does naive "retrieve once, read" struggle?

> 🧠 **Analogy**
>
> **From a lookup to a researcher.** Naive RAG is a clerk who grabs the first folder and reads it aloud. Self-RAG is a clerk who asks "do I even need to look this up? is this folder actually relevant?" before answering. CRAG notices the folder is weak and goes to the web instead. GraphRAG built a map of how every entity connects, so it can answer "what links these cases?" that no single folder holds. Agentic RAG is a full researcher who plans, searches, re-searches, and decides when they have enough.
> **Where this breaks down:** each upgrade adds latency, cost, and moving parts. Start naive-plus-hybrid; add these only when a measured failure mode demands it. A knowledge graph you do not need is just an expensive liability.

> 📦 **Info**
>
> 🤖 RAG evolution taxonomy
> - **Naive RAG:** Linear retrieve → read. No quality checks. Suffers from low precision, missed docs, no self-correction.
> - **Advanced RAG:** Pre-retrieval (query rewriting, HyDE) + post-retrieval (reranking, context compression). Most production systems need this minimum.
> - **Self-RAG:** the model emits "reflection" tokens to decide WHEN retrieval is even needed and to grade its own output, improving factuality over a naive baseline.
> - **CRAG (Corrective RAG):** a lightweight evaluator scores the retrieved docs; when they are weak, it falls back to web search instead of answering from bad context.
> - **[GraphRAG](https://github.com/microsoft/graphrag) (Microsoft):** extracts a knowledge graph of entities and relations from the corpus, so it can answer holistic questions ("what are the main themes across ALL these documents?") that chunk-retrieval cannot, since no single chunk holds the answer.
> - **Agentic RAG:** an agent in a reason-act-observe loop decides when and how to retrieve, retrieving iteratively for multi-hop questions and routing by complexity (the subject of Lesson 8.10).

#### 💡 What just happened

What just happened?RAG has evolved from a simple pattern to a **family of architectures**. Start with Advanced RAG (hybrid search + reranking). Add Self-RAG/CRAG when retrieval quality varies. Use GraphRAG for knowledge bases with complex entity relationships. Agentic RAG for multi-hop questions requiring iterative retrieval. Each adds complexity - use the simplest architecture that meets your quality bar.

---

## 🎯 Concept 10: India RAG - Indic Languages, Sarvam AI, Government Documents, Legal

### India RAG - Indic Languages, Sarvam AI, Government Documents, Legal

Legacy fonts. Cross-lingual retrieval. IndianKanoon. ₹ cost optimization.

Everything above assumed clean English text. Indian RAG breaks that assumption at every stage, and the failures are unglamorous but decisive: PDFs in pre-Unicode fonts that extract as gibberish, tokenizers that make Hindi 2-4x more expensive, and retrieval that works across scripts unevenly. This step is the field guide for the market most of this course ships to.

You build RAG over Hindi government PDFs and the text extracts as gibberish. What is wrong?

> 🏛️ **Analogy**
>
> **Reading a library where half the books are in a secret code.** Many Indian government and publishing PDFs use legacy fonts (Kruti Dev, Shivaji01) that map Devanagari onto Latin codepoints - so a normal extractor reads confident nonsense, like a book printed in a substitution cipher. You must first detect the code and translate it back (Font2Unicode), or OCR scanned pages (Sarvam Vision), BEFORE any chunking or embedding can help. Then multilingual embeddings (BGE-M3, Cohere v4) and an Indic tokenizer keep retrieval accurate and generation affordable.

> 📦 **Info**
>
> 🇮🇳 India-specific RAG patterns
> - **Document processing:** Classify PDFs (Unicode/legacy/scanned). Sarvam Vision for OCR (23 languages). Font2Unicode for Kruti Dev/Shivaji01. pymupdf4llm for clean Unicode.
> - **Embeddings:** BGE-M3 (dense+sparse, 100+ langs). Cohere v4 (proven for 8 Indian languages on AWS). bge-multilingual-gemma2 (reported by Sarvam to improve recall over BGE-M3).
> - **Tokenization:** generic tokenizers split Indic words into many tokens ("token fertility"), so Hindi can cost several times more per word than English; an Indic-tuned tokenizer (e.g. Sarvam) cuts that fertility and the generation bill sharply.
> - **Legal RAG:** IndianKanoon API (1.4M+ documents, SC/HC/tribunal). NyayaRAG: 56K Supreme Court cases + FAISS + all-MiniLM. LawPal: high reported accuracy with small character-level chunks.
> - **Cross-lingual:** same-script language pairs (Hindi-Marathi, both Devanagari) transfer better than cross-script pairs (Hindi-Tamil); a robust pattern is dual retrieval - dense on the Hindi query plus BM25 on an English translation, merged with RRF.
> - **Evaluation:** use ChrF (character-level F-score, robust to rich morphology) rather than BLEU/ROUGE for Indic output. Benchmarks like IndicGenBench cover dozens of languages; safety agreement across languages is notably low, so evaluate per-language.

#### 💡 What just happened

What just happened?India's RAG landscape is unique: **legacy fonts** make PDF processing the #1 challenge, **token fertility** makes generation 2-4× more expensive, and **cross-script retrieval** has significant accuracy gaps. The stack: Sarvam Vision (OCR) + BGE-M3/Cohere v4 (embeddings) + Sarvam tokenizer (generation) + ChrF evaluation. For legal: IndianKanoon API provides the largest indexed corpus of Indian case law.

> 📦 **Info**
>
> 🔗 Where this goes next (Module 8)
> - **In Lesson 8.2 (Document Processing)** and **8.3 (Embeddings & Vector Stores)** we will come back to steps 4-6 in depth - real PDF pipelines, and production vector databases instead of a Python list.
> - **In Lesson 8.4 (Advanced Retrieval)**, step 7's ladder (hybrid, reranking, query transforms) becomes hands-on code.
> - **In Lesson 8.10 (Agentic RAG)** and **8.11 (RAG Evaluation & RAGAS)**, step 9's advanced family and the eval discipline get full lessons - and Lessons 8.5/8.6 rebuild all of this on LangChain and LlamaIndex.

## Key Takeaways

> ✅ **Info**
>
> What you learned
> - **RAG = open-book exam.** Retrieve relevant chunks, inject them into the prompt, generate a grounded answer. No fine-tuning, no GPU - update knowledge by swapping documents.
> - **The 5-step pipeline:** Chunk to Embed to Retrieve to Augment to Generate. You built all five by hand in ~60 lines, so no framework is a black box.
> - **Chunk size is the #1 lever.** Too small = fragments; too large = noise. Recursive splitting around 512 tokens with overlap is the sane default; parent-child solves precision-vs-context.
> - **Embeddings turn meaning into coordinates.** Cosine similarity finds "return my money" near "refund policy" with zero shared words. Switching embedding models means re-embedding the whole corpus - choose deliberately.
> - **"Answer ONLY from context"** is the instruction that prevents hallucination - and lets the model say "I do not know" instead of inventing.
> - **The retrieval ladder:** dense to hybrid (+BM25, merged by RRF) to reranking (cross-encoder) to contextual retrieval. The retriever sets the ceiling - fix retrieval before you fix the reranker.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **RAG Fundamentals- Build It From Scratch**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.1.html` - regenerate this notebook whenever the source HTML is updated.*
